# Notebook 0 — From MATLAB to a Tidy CSV

**Goal.** Convert the proprietary `data/raw/GEIS.mat` measurement file produced by the
lab equipment into a single tidy CSV at `data/processed/batteries_cleaned_dataset.csv`,
so that the rest of the project (notebooks, FastAPI bridge, Streamlit pages, drift
monitoring) can read everything through `pandas.read_csv` without ever touching
scipy or MATLAB structures again.

**Source.** The MATLAB file ships an `Aging{0..4}.GEIS_<temperature>` nested
structure with one impedance matrix per State of Charge. Each row is
`[Frequency, Z_real, Z_imag]` in Ω.

**Output schema** (≈ 9 805 rows, 6 columns):

| Column | Type | Range / unit |
|---|---|---|
| `Aging` | int | `0` (fresh) → `4` (aged) |
| `Temperature` | float | 8 levels in °C: 20.0, 22.5, 25.0, 27.5, 30.0, 35.0, 40.0, 47.5 |
| `SOC` | int | `0` → `4` (≈ 0 %, 25 %, 50 %, 75 %, 100 %) |
| `Frequency` | float | Hz, log-spaced from ≈ 0.099 Hz to ≈ 10 002 Hz (~49 points) |
| `Z_real` | float | Re(Z) in **mΩ** (paper convention; raw MATLAB uses Ω, we multiply by 1 000) |
| `Z_imag` | float | Im(Z) in **mΩ** |

**Equivalent production code.** The same conversion lives in
`src/data/preprocessing.py::mat_to_dataframe`, exercised by
`tests/test_preprocessing.py` and runnable via `make data`. The notebook is kept
as a *pedagogical mirror* so the reader can step through the parsing logic
interactively before invoking the production pipeline.

**Sanity checks to run after writing the CSV.**

1. `df.shape == (9805, 6)`
2. `df.isna().sum().sum() == 0`
3. `df.groupby(["Aging", "Temperature", "SOC"]).ngroups == 200` (40 (Aging, Temp) pairs × 5 SOC levels)
4. `df["Frequency"].between(0.099, 10_002).all()`


In [6]:
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1. Load the raw MATLAB file
mat_data = scipy.io.loadmat('../data/raw/GEIS.mat', squeeze_me=True, struct_as_record=False)

# 2. Initialize a list to collect the records
records = []

# Mapping of file keys to the actual temperature values
temperatures = {
    'GEIS_20': 20.0, 'GEIS_22_5': 22.5, 'GEIS_25': 25.0, 
    'GEIS_27_5': 27.5, 'GEIS_30': 30.0, 'GEIS_35': 35.0, 
    'GEIS_40': 40.0, 'GEIS_47_5': 47.5
}

# 3. Extract data from the MATLAB structure
for aging_idx in range(5):
    aging_key = f'Aging{aging_idx}'
    aging_struct = getattr(mat_data, aging_key, mat_data[aging_key])
    
    for temp_key, temp_val in temperatures.items():
        data = getattr(aging_struct, temp_key)
        for soc_idx, d in enumerate(data):
            for freq_idx, row in enumerate(d):
                freq = row[0]
                z_real = row[1] * 1000 # Convert to mOhm as in the paper
                z_imag = row[2] * 1000 # Convert to mOhm
                
                records.append({
                    'Aging': aging_idx,
                    'Temperature': temp_val,
                    'SOC': soc_idx,  # 0 to 4 (approximately 0%, 25%, 50%, 75%, 100%)
                    'Frequency': freq,
                    'Z_real': z_real,
                    'Z_imag': z_imag
                })

# 4. Build the cleaned DataFrame
df = pd.DataFrame(records)

# Save as CSV for future use / backup
df.to_csv('../data/processed/batteries_cleaned_dataset.csv', index=False)

# Display the first rows of the cleaned dataset
display(df.head())